#### Imports

In [1]:
# Imports
import os
import sys
import cv2
from PIL import Image
from tqdm import tqdm
import supervision as sv
from rfdetr import RFDETRMedium

#### Path Configurations

In [2]:
# Get the absolute path to the 'project' directory
project_root = os.path.abspath(os.path.join('..', '..','..'))

# Add it to sys.path if it's not already there
if project_root not in sys.path:
    sys.path.append(project_root)

# Path Configuration
result_directory = os.path.join(project_root, "results")

# Model Name and Weights
base_model_name = "rfdetr_m"
model_name = "04-05-2026_08-51_rfdetr_m"
model_directory = os.path.join(project_root , "models" , "detection", model_name)
full_model_weights_path = os.path.join(model_directory, "checkpoint_best_ema.pth")

# Test File
file_name = "0bfacc_0"
dataset_name = "roboflow_samples"
data_directory = os.path.join(project_root, "data")
raw_video_data_directory = os.path.join(data_directory, "raw_videos")
full_video_file_path = os.path.join(raw_video_data_directory, dataset_name, f"{file_name}.mp4")

# Results Path
full_result_output_path = os.path.join(result_directory, f"{file_name}_{model_name}_detections.mp4")

#### Image Processor and Model Setup

In [3]:
# Get mapping from category id to category name
categories = ['ball', 'goalkeeper', 'player', 'referee']
id2label = {index: x for index, x in enumerate(categories, start=0)}
label2id = {v: k for k, v in id2label.items()}

# Detection Model
detection_model = RFDETRMedium(pretrain_weights=full_model_weights_path, num_classes = 4)
detection_model.optimize_for_inference()

[2026-05-05 09:43:47] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-05-05 09:43:47] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


#### Ball, Goalkeeper, Players, Referees Coloring

In [4]:
box_annotator = sv.BoxAnnotator(
    color=sv.ColorPalette.from_hex(['#FF8C00', '#00BFFF', '#FF1493', '#FFD700']),
    thickness=2
)
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(['#FF8C00', '#00BFFF', '#FF1493', '#FFD700']),
    text_color=sv.Color.from_hex('#000000')
)

#### Video Annotation Loop

In [5]:
# Video Annotation Setup
video_info = sv.VideoInfo.from_video_path(full_video_file_path)
video_sink = sv.VideoSink(full_result_output_path, video_info=video_info)
frame_generator = sv.get_video_frames_generator(full_video_file_path)

# Video Annotation Loop
with video_sink:
    for frame in tqdm(frame_generator, total=video_info.total_frames):

        # Run inference — rfdetr expects a PIL Image
        pil_frame = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        detections = detection_model.predict(pil_frame, threshold=0.3)
        # detections is already a sv.Detections object

        labels = [
            f"{id2label[class_id]} {confidence:.2f}"
            for class_id, confidence
            in zip(detections.class_id, detections.confidence)
        ]

        annotated_frame = frame.copy()
        annotated_frame = box_annotator.annotate(scene=annotated_frame, detections=detections)
        annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)
        video_sink.write_frame(annotated_frame)

100%|██████████| 750/750 [01:02<00:00, 11.95it/s]
